# yolo26_cpp — Thrust device (GPU) check: parity + training + cuDNN
Device-resident engine (`dtensor.hpp`/`dnet26.hpp`) is CPU-verified + nvcc-compiles; this runs it
on a real GPU. **Runtime → GPU (T4)**, then Run all. (yolo26 = yolo11 body + SPPF residual +
L22 Bottleneck+PSABlock + no-DFL end2end head.)


In [ ]:
!nvidia-smi -L
!nvcc --version | tail -1


In [ ]:
%cd /content
!rm -rf yolo26_cpp
!git clone -q https://github.com/yomei-o/yolo26_cpp.git
%cd /content/yolo26_cpp
!pip -q install ultralytics


### Export refs + init weights (pure C++/Python, no run-time deps)


In [ ]:
!python pure/ref/export_yolo26.py 64 yolo26n         # fused refs + weights.bin (auto-uses yolo26n.pt)
!python pure/ref/export_unfused26.py 64 yolo26n      # unfused manifest/names + arch26 (pretrained bins)
!g++ -O2 -std=c++17 -Ipure/third_party pure/make_init_pt.cpp -o make_init_pt
!./make_init_pt init26.pt rand x pure/ref/data_net/  # random init for the parity test


### Locate cuDNN (for step 3) — auto


In [ ]:
import os, glob
inc=lib=None
try:
    import nvidia.cudnn; d=os.path.dirname(nvidia.cudnn.__file__)
    if os.path.exists(d+"/include/cudnn.h"): inc,lib=d+"/include",d+"/lib"
except Exception: pass
if not inc:
    for h in ["/usr/include/cudnn.h"]+glob.glob("/usr/include/**/cudnn.h",recursive=True)+glob.glob("/usr/local/cuda*/include/cudnn.h"):
        if os.path.exists(h): inc,lib=os.path.dirname(h),"/usr/lib/x86_64-linux-gnu"; break
os.environ["CUDNN_INC"],os.environ["CUDNN_LIB"]=inc or "",lib or ""
print("CUDNN_INC =",inc,"\nCUDNN_LIB =",lib)


### 1. Device forward parity on GPU (dnet26_test) — expect MATCH


In [ ]:
!nvcc -x cu -O2 -std=c++17 --extended-lambda -arch=native -DUSE_CUDA -Ipure/third_party pure/dnet26_test.cpp -o dnet26_gpu
!./dnet26_gpu


**Note:** device train-mode vs CPU was ~6e-2 at P5 on CPU-thrust (batch-stat reduction order + double attention). If GPU is much worse, inspect d26_c3k2_psa/d26_sppf.


### 2. Device training on GPU (dtrain_coco26) — COCO128, from pretrained


In [ ]:
!wget -q https://github.com/ultralytics/yolov5/releases/download/v1.0/coco128.zip && unzip -q -o coco128.zip
!./make_init_pt init26_pre.pt from yolo26n.pt pure/ref/data_net/    # pretrained init
!nvcc -x cu -O2 -std=c++17 --extended-lambda -arch=native -DUSE_CUDA -DUSE_CUBLAS -Ipure/third_party pure/dtrain_coco26.cpp -lcublas -o dtrain26
!./dtrain26 coco128/images/train2017 320 8 10 init26_pre.pt pure/ref/data_net/


Expect loss to decrease and a sensible s/epoch (much faster than CPU). (CPU baseline: pretrained val mAP@0.5 0.504; CPU fine-tune reached 0.540.)


### 3. cuDNN device path (grouped) — dnet26_test with -DUSE_CUDNN


In [ ]:
!nvcc -x cu -O2 -std=c++17 --extended-lambda -arch=native -DUSE_CUDA -DUSE_CUDNN \
      -I"$CUDNN_INC" -L"$CUDNN_LIB" -Ipure/third_party pure/dnet26_test.cpp -lcudnn -o dnet26_cudnn
!LD_LIBRARY_PATH="$CUDNN_LIB:$LD_LIBRARY_PATH" ./dnet26_cudnn


All three should run on the T4. This is the only part of yolo26_cpp not yet verified on real GPU hardware.
